<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Processing Fundamental</b></h1>
</div>

## Theoretical Foundations

This notebook documents the mathematical model, estimation methods, numerical considerations, diagnostics, and limitations used in the laboratory.
### Technical Context

Digital-image processing treats visual information as discrete numerical arrays.

### Core Digital Image Model

A grayscale image is a discrete function $I[y,x]$; a color image extends the array with a channel dimension $I[y,x,c]$. Sampling discretizes space and quantization discretizes intensity.

### Notation and Conventions

| Symbol | Meaning |
| --- | --- |
| $I$ | image array |
| $x$ | column coordinate |
| $y$ | row coordinate |
| $c$ | channel index |
| $H,W$ | image height and width |
| $L$ | number of intensity levels |

### Analytical Scope

Explain image representation, manipulate pixel values safely, interpret statistics and histograms, model common degradations, compare images numerically, and validate results.


## 1. Data and Output Paths

The notebook is stored under `notebooks/`, while the input images are stored under `data/`.

The path resolver below walks upward from the current working directory until it finds the laboratory root. This makes the notebook robust when VS Code or Jupyter starts from slightly different working directories.


## 2. Sampling and Quantization Experiments

A real-world scene is continuous in space and light intensity. A digital image is not continuous: it is a finite grid of measurements.

A simplified acquisition chain is:

```text
Real scene
    ↓
Optical system
    ↓
Image sensor
    ↓
Spatial sampling
    ↓
Intensity quantization
    ↓
Digital image array
```

Two concepts are fundamental:

- **Sampling** decides *where* measurements are taken in space.
- **Quantization** decides *which numerical values* can represent the measured intensity.


## 3. Pixel Coordinates and Array Representation

A **pixel** is one spatial sample of a digital image.

A grayscale image can be represented as a 2-D array:

$$
I[y,x]
$$

where:

- `y` = row index;
- `x` = column index;
- the stored number is the pixel intensity.

In image-processing mathematics we often write $I(x,y)$, but NumPy uses:

```python
image[row, column]
image[y, x]
```

This distinction is extremely important.


## 4. Image Representation Modes

Three basic image types appear constantly in image processing.

### Binary image

Contains two logical states, often:

- 0 = background;
- 1 or 255 = foreground.

### Grayscale image

Contains one intensity value per pixel.

Typical shape:

```text
(H, W)
```

### RGB image

Contains three values per pixel:

```text
R = red
G = green
B = blue
```

Typical shape:

```text
(H, W, 3)
```


## 5. Load and Inspect Real Images

Before processing an unfamiliar image, inspect:

1. shape;
2. dtype;
3. minimum and maximum values;
4. number of channels;
5. visual appearance.

This avoids many silent errors.


## 6. Dimensions, Resolution, Aspect Ratio, and Channels

For an RGB array with:

```python
image.shape == (H, W, 3)
```

- `H` = height in pixels;
- `W` = width in pixels;
- `3` = number of channels.

The total number of pixels is:

$$
N = H \times W
$$

The aspect ratio is commonly:

$$
\text{aspect ratio} = \frac{W}{H}
$$


## 7. Data Types, Bit Depth, Dynamic Range, and Memory

A NumPy image has a data type (`dtype`).

Typical examples:

| dtype | Typical use | Example range |
|---|---|---|
| `bool` | binary logic | `False`, `True` |
| `uint8` | standard images | 0–255 |
| `uint16` | higher bit-depth imaging | 0–65535 |
| `float32` | processing / ML | often 0–1 |
| `float64` | numerical calculations | application-dependent |

For an unsigned 8-bit integer:

$$
0 \le I \le 255
$$

because:

$$
2^8 = 256
$$

possible values are available.


## 8. Display Scaling and Visualization Control

Displaying an array is not always trivial.

For grayscale images, Matplotlib may automatically rescale values unless `vmin` and `vmax` are specified.

For fair visual comparison, explicit display limits are often necessary.


## 9. Pixel Access and Safe Modification

A color pixel is a vector:

```python
[R, G, B]
```

To inspect one pixel:

```python
pixel = image[y, x]
```

To modify an image while keeping the original unchanged, use:

```python
copy = image.copy()
```


## 10. Regions of Interest (ROI)

A **region of interest** is a selected subregion of an image.

With NumPy slicing:

```python
roi = image[y_start:y_end, x_start:x_end]
```

The vertical range comes first because arrays are indexed as:

```python
[row, column]
```


## 11. Pixel Neighborhoods

Many image-processing operations depend on a pixel **and its neighbors**.

A common $3\times3$ neighborhood around pixel $(x,y)$ is:

$$
\begin{bmatrix}
I(y-1,x-1) & I(y-1,x) & I(y-1,x+1) \\
I(y,x-1)   & I(y,x)   & I(y,x+1)   \\
I(y+1,x-1) & I(y+1,x) & I(y+1,x+1)
\end{bmatrix}
$$

This idea becomes essential in the spatial-filtering lab.


## 12. RGB Channel Decomposition

An RGB image can be interpreted as three separate 2-D planes:

$$
I_R(y,x), \qquad I_G(y,x), \qquad I_B(y,x)
$$

In NumPy:

```python
red   = image[..., 0]
green = image[..., 1]
blue  = image[..., 2]
```


## 13. RGB and BGR Conventions

Different libraries may use different channel orders.

- **Pillow** → RGB
- **Matplotlib** → expects RGB
- **OpenCV** → traditionally loads color images as BGR

If BGR data is displayed as RGB, red and blue are exchanged.


## 14. RGB-to-Grayscale Conversion

A simple arithmetic mean:

$$
Y = \frac{R+G+B}{3}
$$

treats all channels equally.

A common luminance approximation gives different weights:

$$
Y = 0.299R + 0.587G + 0.114B
$$

because human vision is more sensitive to green than to blue.


## 15. Image Statistics

Useful scalar summaries include:

- minimum;
- maximum;
- mean;
- median;
- standard deviation;
- percentiles.

These statistics help characterize global brightness and intensity spread.


## 16. Intensity Histograms

A grayscale histogram counts how many pixels belong to each intensity bin.

For an 8-bit image, a natural choice is 256 bins representing values 0–255.

A histogram can indicate whether an image is:

- globally dark;
- globally bright;
- low contrast;
- spread over a wide dynamic range.


## 17. Dynamic Range and Min-Max Normalization

If an image uses only a narrow intensity interval, contrast may be weak.

Min-max normalization maps:

$$
I_{\min} \rightarrow 0
$$

and:

$$
I_{\max} \rightarrow 255
$$

using:

$$
I_{\mathrm{norm}}
=
\frac{I-I_{\min}}{I_{\max}-I_{\min}}
\times 255
$$


## 18. `uint8` Arithmetic, Overflow, Clipping, and Floating Point

This section addresses numerical safety for integer-valued image arithmetic.

`uint8` can store only 0–255. Values outside this range cannot be represented.

For safe arithmetic:

1. convert to a wider or floating-point type;
2. perform the operation;
3. clip to the valid output range;
4. convert back if necessary.


## 19. Noise Model Simulation

Noise is unwanted variation in measured pixel values.

Sources include:

- sensor electronics;
- low-light acquisition;
- photon statistics;
- transmission;
- compression;
- environmental interference.

Common noise models include:

### Gaussian noise
Additive continuous fluctuations.

### Salt-and-pepper noise
Random isolated dark and bright pixels.

### Poisson noise
Signal-dependent counting noise.

### Speckle noise
Multiplicative granular noise.


## 20. Image Comparison Metrics

Suppose $R$ is a reference image and $T$ is a test image.

### Mean Absolute Error (MAE)

$$
\mathrm{MAE}
=
\frac{1}{N}
\sum |R-T|
$$

### Mean Squared Error (MSE)

$$
\mathrm{MSE}
=
\frac{1}{N}
\sum (R-T)^2
$$

### Root Mean Squared Error (RMSE)

$$
\mathrm{RMSE}
=
\sqrt{\mathrm{MSE}}
$$

### Peak Signal-to-Noise Ratio (PSNR)

For 8-bit images:

$$
\mathrm{PSNR}
=
10\log_{10}
\left(
\frac{255^2}{\mathrm{MSE}}
\right)
$$

Higher PSNR usually means greater numerical similarity to the reference.


## 21. Lossless vs Lossy Image Encoding

### PNG

- lossless compression;
- preserves pixel values;
- appropriate for masks, diagrams, labels, and quantitative intermediate results.

### JPEG

- lossy compression;
- efficient for natural photographs;
- decoded values may differ from the original;
- repeated save/load cycles may introduce artifacts.

For quantitative processing, prefer lossless formats unless compression is part of the experiment.


## 22. Standard Image Inspection Workflow

Whenever you receive an unfamiliar image, use this sequence:

```text
1. Locate the file
2. Load it
3. Inspect shape
4. Inspect dtype
5. Inspect min / max
6. Determine color/channel convention
7. Display it correctly
8. Compute useful statistics
9. Decide whether conversion is needed
10. Process in a safe numeric dtype
11. Validate the result numerically
12. Validate the result visually
13. Save outputs reproducibly
```

This simple workflow prevents many silent bugs.


## 23. Validation Checks

A good notebook should verify important assumptions explicitly rather than relying only on visual inspection.


## Technical Synthesis

The laboratory establishes the numerical representation on which all later image-processing modules depend:

$$
\boxed{
\text{scene}
\rightarrow
\text{sampling/quantization}
\rightarrow
\text{image array}
\rightarrow
\text{channels/statistics}
\rightarrow
\text{histogram/noise}
\rightarrow
\text{quantitative comparison}
}
$$

The essential engineering constraints are array shape, coordinate convention, channel order, dtype, dynamic range, numerical safety, and metric interpretation. Errors in these foundations propagate directly into filtering, transformation, segmentation, and computer-vision pipelines.

## Scope and Limitations

### Included

Digital-image representation, inspection, pixel/channel operations, histograms, normalization, noise models, numerical comparison, saving, and validation.

### Not included

Geometric transformations, spatial/frequency filtering, and segmentation algorithms.
